# Import libs

In [ ]:
# import tensorflow and keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras.backend as K

# import numpy to create arrays for trial runs
import numpy as np
# matplotlib for visualisations
import matplotlib.pyplot as plt
# if using Colab, for later saving of models and loading data
from google.colab import drive
drive.mount('/content/drive')
from torch.utils.data import DataLoader, Dataset, Subset
from sklearn.model_selection import train_test_split
from pathlib import Path


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


: 

In [2]:
#!unzip -q /content/drive/MyDrive/BMET5933/WEEK_10/hep2imgcnn.zip -d /content/drive/MyDrive/BMET5933/WEEK_10

In [2]:
# dataset path config
dataset_path = Path("/content/drive/MyDrive/BMET5933/WEEK_10/hep2imgcnn")

# get class names and create mapping to labels
class_names = sorted([folder.name for folder in dataset_path.iterdir() if folder.is_dir()])
class_to_label = {class_name: idx for idx, class_name in enumerate(class_names)}
print("Class to label mapping:", class_to_label)

image_paths = []
labels = []

# iterate through each class directory and collect image paths and labels
dircetory_names = [folder.name for folder in dataset_path.iterdir() if folder.is_dir()]
for directory in dircetory_names:
	folder_path = dataset_path / directory
	image_paths = list(folder_path.glob("*.png"))

	# store all the image paths and corresponding labels
	for image_path in image_paths:
		image_paths.append(image_path)
		labels.append(class_to_label[directory])
		

Class to label mapping: {'centromere': 0, 'coarse_speckled': 1, 'fine_speckled': 2, 'homogeneous': 3, 'nucleolar': 4}


: 

: 

In [ ]:
SEED=42
BATCH_SIZE=32
IMAGE_DIR=str(dataset_path) # this is the path to the directory containing the class subdirectories
RESCALED_IMAGE_SIZE=(51,51) # images will be all resized to this - they will have 3 channels

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    IMAGE_DIR,
    batch_size=BATCH_SIZE,
    image_size=RESCALED_IMAGE_SIZE,
	shuffle=True,
	seed = SEED,
    validation_split=0.3,
    subset='training'
)

validation_ds = tf.keras.preprocessing.image_dataset_from_directory(
	IMAGE_DIR,
	labels='inferred',
	label_mode='int',
	batch_size=BATCH_SIZE,
	image_size=RESCALED_IMAGE_SIZE,
	shuffle=True,
	seed = SEED,
	validation_split=0.3,
	subset='validation'
)

# you can optimise data loading with prefetching 
PREFETCH_SIZE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=PREFETCH_SIZE)
validation_ds = validation_ds.prefetch(buffer_size=PREFETCH_SIZE)


Found 453 files belonging to 5 classes.
Using 363 files for training.
Found 453 files belonging to 5 classes.
Using 90 files for validation.


In [ ]:
# define a shallow CNN model
def create_shallow_cnn(input_shape, num_classes):
	model = keras.Sequential([
		layers.Input(shape=input_shape),
		layers.Conv2D(16, kernel_size=(5, 5), activation='relu'),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Flatten(),
		layers.Dense(num_classes, activation='softmax')
	])
	return model

# initialise model and print summary
shallow_cnn = create_shallow_cnn(input_shape=(51, 51, 3), num_classes=len(class_names))

# check model architecture
shallow_cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 49, 49, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 22, 22, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 11, 11, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 7744)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       991,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,011,397 (3.86 MB)

 Trainable params: 1,011,397 (3.86 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# define a deeper CNN
def create_deeper_cnn(input_shape, num_classes):
	model = keras.Sequential([
		layers.Input(shape=input_shape),
		layers.Conv2D(32, kernel_size=(3, 3), activation='relu'),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Conv2D(128, kernel_size=(3, 3), activation='relu'),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Flatten(),
		layers.Dense(64, activation='relu'),
		layers.Dense(num_classes, activation='softmax')
	])
	return model

# initialise deeper model and print summary
deeper_cnn = create_deeper_cnn(input_shape=(51, 51, 3), num_classes=len(class_names))

# check deeper model architecture
deeper_cnn.summary()

In [ ]:
LEARNING_RATE = 1e-4 # Some common values are 1e-3 (0.001) or 1e-5 (0.00001)
BATCH_SIZE = 32 # train with this many images per iteration [5, 10, or 20 might be good for a trial run]
NUM_EPOCHS = 25 # how many epochs to train for (each epoch visits the training data once) [5 might be good for a trial run]

# compile the model
model.compile(
    optimizer=keras.optimizers.Adam(LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# train_history will store the metrics for each epoch, for use in generating graphs
train_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=NUM_EPOCHS
)

Epoch 1/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 167ms/step - accuracy: 0.5289 - loss: 1.3738 - val_accuracy: 0.5778 - val_loss: 0.9597
Epoch 2/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 158ms/step - accuracy: 0.6970 - loss: 0.7130 - val_accuracy: 0.6444 - val_loss: 0.9725
Epoch 3/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 161ms/step - accuracy: 0.7107 - loss: 0.6243 - val_accuracy: 0.7000 - val_loss: 0.9051
Epoch 4/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 242ms/step - accuracy: 0.7410 - loss: 0.5618 - val_accuracy: 0.6000 - val_loss: 0.9658
Epoch 5/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 149ms/step - accuracy: 0.7989 - loss: 0.4944 - val_accuracy: 0.7000 - val_loss: 0.7766
Epoch 6/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 146ms/step - accuracy: 0.8072 - loss: 0.4850 - val_accuracy: 0.6444 - val_loss: 0.7165
Epoch 7/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 148ms/step - accuracy: 0.8154 - loss: 0.4724 - val_accuracy: 0.7111 - val_loss: 0.7655
Epoch 8/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 161ms/step - accuracy: 0.7934 - loss: 0.4827 - val_accuracy: 0.

In [ ]:
# evaluatin on the validation set
val_loss, val_acc = model.evaluate(val_ds)
print(f"Validation loss: {val_loss:.4f}, Validation accuracy: {val_acc:.4f}")

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.7333 - loss: 0.5991
Validation loss: 0.5991, Validation accuracy: 0.7333
